In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import time and json
# 2. Define make_jsonrpc_request(method, params=None, req_id=None) -> dict:
#    Return {"jsonrpc": "2.0", "method": method, "params": params or {}, "id": req_id}
# 3. Define make_jsonrpc_response(result=None, error=None, req_id=None) -> dict:
#    If error is not None include "error" key; else include "result" key
# 4. Smoke-test: call make_jsonrpc_request("tools/list", req_id="1")
#    and print(json.dumps(..., indent=2)) to verify JSON-RPC 2.0 format
#
# Hint:
#   def make_jsonrpc_request(method, params=None, req_id=None):
#       return {"jsonrpc": "2.0", "method": method, "params": params or {}, "id": req_id}
#
#   def make_jsonrpc_response(result=None, error=None, req_id=None):
#       base = {"jsonrpc": "2.0", "id": req_id}
#       if error is not None:
#           base["error"] = error
#       else:
#           base["result"] = result
#       return base
#
#   req = make_jsonrpc_request("tools/list", req_id="1")
#   print(json.dumps(req, indent=2))

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import json, jsonschema
# 2. Define custom exceptions: ToolNotFoundError, ToolExecutionError(code, message)
# 3. Define class MCPClient with:
#    - __init__: self._tool_registry = {}, self._next_id = 1
#    - next_request_id(self) -> str: return str(self._next_id) and increment
# 4. Add discover_tools(self, response: dict):
#    Iterate response["result"]["tools"], store each tool's inputSchema in
#    self._tool_registry keyed by tool name; print discovered names
# 5. Add get_tool_schema(self, tool_name: str) -> dict:
#    Raise ToolNotFoundError if not found; else return cached schema
# 6. Simulate a tools/list response with two tools (get_supplier_quote, send_purchase_order)
#    Call discover_tools() and print the schema for get_supplier_quote
#
# Hint:
#   class ToolExecutionError(Exception):
#       def __init__(self, code, message): self.code = code; self.message = message
#
#   class MCPClient:
#       def __init__(self):
#           self._tool_registry: dict = ???
#           self._next_id: int = ???
#       def discover_tools(self, response):
#           for tool in response["result"]["tools"]:
#               self._tool_registry[tool["name"]] = ???   # store inputSchema
#       def get_tool_schema(self, tool_name):
#           if tool_name not in self._tool_registry:
#               raise ToolNotFoundError(???)
#           return self._tool_registry[???)
#
#   client = MCPClient()
#   client.discover_tools(tools_list_response)
#   print(json.dumps(client.get_tool_schema("get_supplier_quote"), indent=2))

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define call_tool(client, tool_name, arguments) -> dict:
#    a. Retrieve schema: client.get_tool_schema(tool_name)
#    b. Validate arguments against schema using jsonschema.validate()
#       Raise ValueError on ValidationError
#    c. Build a JSON-RPC 2.0 "tools/call" request dict
#    d. Simulate a server response (hardcode result for DESK-001 / TechFurnish)
#    e. Return result dict or raise ToolExecutionError if response has "error"
# 2. Call call_tool() for PO #2024-1847:
#    tool="get_supplier_quote", supplier="TechFurnish", item="DESK-001", qty=10
#    Print the returned price and delivery_days
# 3. Call it with invalid arguments (e.g. quantity=-5) to trigger ValidationError
#
# Hint:
#   def call_tool(client, tool_name, arguments):
#       schema = client.get_tool_schema(???)
#       try:
#           jsonschema.validate(instance=???, schema=???)
#       except jsonschema.ValidationError as e:
#           raise ValueError(f"Invalid arguments for '{tool_name}': {e.message}")
#
#       request = make_jsonrpc_request("tools/call",
#                     params={"name": tool_name, "arguments": arguments},
#                     req_id=client.next_request_id())
#       simulated_response = make_jsonrpc_response(
#           result={"price": ???, "delivery_days": ???, "currency": "USD"},
#           req_id=request["id"]
#       )
#       if "result" in simulated_response:
#           return simulated_response["result"]
#       raise ToolExecutionError(code=???, message=???)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define RETRYABLE_CODES = {-32603} and NO_RETRY_CODES = {-32700, -32600, -32601, -32602}
# 2. Define call_tool_with_retry(client, tool_name, arguments,
#                                max_retries=3, base_delay=1.0) -> dict:
#    a. Loop up to max_retries attempts
#    b. On success: return the result
#    c. On ToolExecutionError with code in RETRYABLE_CODES (-32603):
#       delay = base_delay * (2 ** attempt)   # exponential backoff
#       print warning with attempt number and delay; time.sleep(delay)
#    d. On ToolExecutionError with code in NO_RETRY_CODES: re-raise immediately
#    e. After exhausting retries: raise ToolExecutionError(-1, "max retries exceeded")
# 3. Simulate transient failure: raise ToolExecutionError(-32603) on attempts 0 and 1,
#    then succeed on attempt 2
# 4. Call call_tool_with_retry() and print final result + total attempts taken
# 5. Simulate non-retryable error (code -32602) and confirm no retry occurs
#
# Hint:
#   NO_RETRY_CODES = {-32700, -32600, -32601, -32602}
#
#   def call_tool_with_retry(client, tool_name, arguments, max_retries=???, base_delay=???):
#       for attempt in range(???):
#           try:
#               return call_tool(client, tool_name, arguments)
#           except ToolExecutionError as e:
#               if e.code == ???:    # -32603 internal error
#                   delay = base_delay * (??? ** attempt)
#                   print(f"Transient error on attempt {attempt+1}, retrying in {delay}s")
#                   time.sleep(delay)
#               else:
#                   raise
#       raise ToolExecutionError(???, "max retries exceeded")

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Simulate all 4 MCP phases end-to-end for OrderFlow PO #2024-1847:
#
#    Phase 1 - INITIALIZE:
#    Build initialize request (method="initialize") with params:
#      protocolVersion="2024-11-05", clientInfo={"name": "OrderFlow-PricingAgent"}
#    Build server capabilities response (tools: {listChanged: true})
#    Validate protocolVersions match; print "Handshake OK"
#
#    Phase 2 - DISCOVER:
#    Create fresh MCPClient, call discover_tools() with tools/list response
#    Print discovered tool count and names
#
#    Phase 3 - CALL:
#    Call call_tool_with_retry() for get_supplier_quote:
#      supplier="TechFurnish", item="DESK-001", qty=10
#    Print returned price and delivery_days
#
#    Phase 4 - AUDIT LOG:
#    Build structured audit dict: po_id, tool_name, arguments, result, timestamp
#    Print as formatted JSON
#
# 2. Print a before/after summary comparing bespoke integrations vs MCP:
#    integrations: 160 -> 28 | error_rate: 3.8% -> 3.2% | onboarding: 2w -> 2d
#
# Hint:
#   # Phase 1
#   init_req  = make_jsonrpc_request("initialize",
#                   params={"protocolVersion": ???, "clientInfo": {???}})
#   init_resp = make_jsonrpc_response(
#                   result={"protocolVersion": ???,
#                           "capabilities": {"tools": {"listChanged": ???}}},
#                   req_id=init_req["id"])
#   assert init_req["params"]["protocolVersion"] == init_resp["result"]["protocolVersion"]
#
#   # Phase 3
#   result = call_tool_with_retry(client, "get_supplier_quote",
#                {"supplier_name": ???, "item_id": ???, "quantity": ???})
#
#   # Phase 4
#   audit_record = {"po_id": ???, "tool_name": ???, "result": ???, "timestamp": ???}
#   print(json.dumps(audit_record, indent=2))